Back test util

In [6]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import backtrader as bt
from datetime import datetime

In [9]:

# import inspect
# inspect.getfile(bt.Indicator)

'd:\\ProgramData\\Anaconda3\\lib\\site-packages\\backtrader\\indicator.py'

In [ ]:
def format_value(value, format_str=".2f", default=0):
    """安全格式化数值"""
    if value is None:
        return format(default, format_str)
    try:
        return format(value, format_str)
    except:
        return format(default, format_str)

In [ ]:
def safe_get_analysis(analyzer, default=0):
    """安全获取分析结果"""
    try:
        analysis = analyzer.get_analysis()
        return analysis
    except:
        return {}

In [8]:
class StopLossPlotter(bt.Indicator):
    lines = ('stop_loss_plot',)
    # 关键设置：确保指标显示在主图
    plotinfo = dict(
        subplot=False,  # 不在副图显示，与K线在同一区域
        plotname='Stop Loss Line',
        plotabove=False,  # 不在K线之上显示
        linestyle = '--',
    )
    
    def __init__(self):
        super(StopLossPlotter, self).__init__()
        # 直接使用数据线，不需要参数
        self.lines.stop_loss_plot = self.data.stop_loss_long

In [ ]:
class LongStrategy(bt.Strategy):
    params = (
        ('printlog', True),
    )
    
    def __init__(self):
        # 引用数据中的列
        self.dataclose = self.datas[0].close
        self.long_signal = self.datas[0].long
        self.stop_loss_long = self.datas[0].stop_loss_long
        self.sell_signal = self.datas[0].sell
        
        # 跟踪变量
        self.order = None
        self.buyprice = None
        self.buycomm = None
 
        # 用于计算年化收益
        self.start_value = None
        self.end_value = None
        self.trade_days = None
        
        self.stop_loss_plotter = StopLossPlotter()

#         # 设置线条样式，让它在K线图上更明显
#         self.stop_loss_plotter.lines.stop_loss_plot.plotinfo.color = 'red'      # 红色
#         self.stop_loss_plotter.lines.stop_loss_plot.plotinfo.linestyle = '--'   # 虚线
#         self.stop_loss_plotter.lines.stop_loss_plot.plotinfo.linewidth = 2      # 线宽 

#     def log(self, txt, dt=None, doprint=False):
#         '''日志函数'''
#         if self.params.printlog or doprint:
#             dt = dt or self.datas[0].datetime.date(0)
#             print(f'{dt.isoformat()}, {txt}')
    
    def log(self, txt, dt=None, doprint=False):
        if self.params.printlog or doprint:
            # 获取当前数据的完整日期时间
            dt = dt or self.datas[0].datetime.datetime(0) 
            # 格式化为 yyyy-mm-dd HH:MM
            print('%s, %s' % (dt.strftime("%Y-%m-%d %H:%M"), txt))
            
            
    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            # 订单已提交/接受 - 无需操作
            return

        # 检查订单是否完成
        if order.status in [order.Completed]:
            if order.isbuy():
                self.log(
                    f'买入执行, 价格: {order.executed.price:.2f}, '
                    f'成本: {order.executed.value:.2f}, '
                    f'佣金: {order.executed.comm:.2f}'
                )
                self.buyprice = order.executed.price
                self.buycomm = order.executed.comm
            elif order.issell():
                self.log(
                    f'卖出执行, 价格: {order.executed.price:.2f}, '
                    f'成本: {order.executed.value:.2f}, '
                    f'佣金: {order.executed.comm:.2f}'
                )
            
            self.bar_executed = len(self)

        elif order.status in [order.Canceled, order.Margin, order.Rejected]:
            self.log('订单 取消/保证金不足/拒绝')

        # 重置订单
        self.order = None

    def notify_trade(self, trade):
        if not trade.isclosed:
            return

        self.log(f'交易利润, 毛利润: {trade.pnl:.2f}, 净利润: {trade.pnlcomm:.2f}')

    def next(self):
        # 如果有未完成的订单，不进行新操作
        if self.order:
            return

        # 检查是否持仓
        if not self.position:
            # 没有持仓，检查买入信号
            if self.long_signal[0] == 1:
                self.log(f'买入信号触发, 价格: {self.dataclose[0]:.2f}')
                # 买入95%的资金
                size = int(self.broker.getcash() * 0.95 / self.dataclose[0])
                if size > 0:
                    self.order = self.buy(size=size)
        
        else:
            # 已有持仓，检查卖出条件
            sell_condition = False
            reason = ""
            
            # 条件1: 止损条件
            if self.dataclose[0] < self.stop_loss_long[0]:
                sell_condition = True
                reason = f"止损, 当前价: {self.dataclose[0]:.2f}, 止损价: {self.stop_loss_long[0]:.2f}"
            
            # 条件2: 卖出信号
            elif self.sell_signal[0] == 1:
                sell_condition = True
                reason = "卖出信号触发"
            
            if sell_condition:
                self.log(f'卖出条件触发: {reason}')
                self.order = self.close()

    def stop(self):
        self.log(f'期末资金: {self.broker.getvalue():.2f}')



In [7]:
class SignalStrategy2(bt.Strategy):
    params = (
        ('printlog', True),
    )

    def __init__(self):
        # 引用数据中的列
        self.open = self.datas[0].open
        self.high = self.datas[0].high
        self.low = self.datas[0].low
        self.close = self.datas[0].close
        self.volume = self.datas[0].volume
        self.long_signal = self.datas[0].long
        self.stop_loss_long = self.datas[0].stop_loss_long
        self.sell_signal = self.datas[0].sell
        
        # 跟踪订单和持仓
        self.order = None
        self.position_size = 0

    def log(self, txt, dt=None, doprint=False):
        '''日志函数'''
        if self.params.printlog or doprint:
            dt = dt or self.datas[0].datetime.date(0)
            print(f'{dt.isoformat()}, {txt}')

    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            # 订单已提交/接受 - 无需操作
            return

        if order.status in [order.Completed]:
            if order.isbuy():
                self.log(f'买入执行, 价格: {order.executed.price:.2f}, 成本: {order.executed.value:.2f}, 佣金: {order.executed.comm:.2f}')
            elif order.issell():
                self.log(f'卖出执行, 价格: {order.executed.price:.2f}, 成本: {order.executed.value:.2f}, 佣金: {order.executed.comm:.2f}')

            self.position_size = order.executed.size

        elif order.status in [order.Canceled, order.Margin, order.Rejected]:
            self.log('订单 取消/保证金不足/拒绝')

        # 重置订单
        self.order = None

    def next(self):
        # 如果有未完成的订单，不进行新操作
        if self.order:
            return

        # 检查是否持仓
        if not self.position:
            # 没有持仓，检查买入信号
            if self.long_signal[0] == 1:
                # 计算买入数量（这里使用全部资金）
                size = int(self.broker.getcash() / self.close[0])
                if size > 0:
                    self.log(f'买入信号, 价格: {self.close[0]:.2f}, 数量: {size}')
                    self.order = self.buy(size=size)
        
        else:
            # 已有持仓，检查卖出条件
            sell_condition = False
            reason = ""
            
            # 条件1: 止损条件
            if self.close[0] < self.stop_loss_long[0]:
                sell_condition = True
                reason = f"止损, 价格: {self.close[0]:.2f}, 止损价: {self.stop_loss_long[0]:.2f}"
            
            # 条件2: 卖出信号
            elif self.sell_signal[0] == 1:
                sell_condition = True
                reason = "卖出信号"
            
            if sell_condition:
                self.log(f'卖出条件触发: {reason}')
                self.order = self.sell(size=self.position.size)

    def stop(self):
        self.log(f'期末资金: {self.broker.getvalue():.2f}')

In [10]:
# 1. 定义一个专门计算年化回报的分析器
class AnnualReturnAnalyzer(bt.Analyzer):
    def __init__(self):
        self.start_value = None
        self.end_value = None
        self.start_date = None
        self.end_date = None

    def start(self):
        # 在回测开始时记录初始资金和日期
        self.start_value = self.strategy.broker.getvalue()
        self.start_date = self.strategy.datas[0].datetime.date(1) # 获取第一个有效数据点日期

    def stop(self):
        # 在回测结束时记录最终资金和日期
        self.end_value = self.strategy.broker.getvalue()
        self.end_date = self.strategy.datas[0].datetime.date(0) # 获取最后一个数据点日期

    def get_analysis(self):
        # 计算总回报
        total_return = (self.end_value / self.start_value) - 1
        
        # 计算回测总天数
        days_total = (self.end_date - self.start_date).days
        # 转换为年数，使用365.25考虑闰年
        years = days_total / 365.25
        
        # 计算年化回报率
        if years > 0:
            annual_return = (self.end_value / self.start_value) ** (1 / years) - 1
        else:
            annual_return = 0
        
        # 返回一个包含结果的字典
        return {
            'start_date': self.start_date,
            'end_date': self.end_date,
            'start_value': self.start_value,
            'end_value': self.end_value,
            'total_return': total_return,
            'annual_return': annual_return,
            'total_days': days_total,
            'total_years': years
        }

In [ ]:
class CustomData(bt.feeds.PandasData):
    """
    自定义数据类，添加额外的列
    """
    lines = ('long', 'stop_loss_long', 'sell',)
    
    params = (
        ('long', -1),      # 如果为-1，表示自动检测列名
        ('stop_loss_long', -1),
        ('sell', -1),
    )

In [ ]:
def calculate_annualized_return(annual_returns):
    """基于AnnualReturn分析器的结果计算年化回报率"""
    if not annual_returns:
        return 0.0
    
    # 获取每年的回报率（小数形式）
    returns = list(annual_returns.values())
    
    # 计算几何平均年化回报率
    product = 1.0
    for ret in returns:
        product *= (1 + ret)
    
    # 年化回报率 = (总乘积)^(1/年数) - 1
    n_years = len(returns)
    annualized_return = (product ** (1 / n_years)) - 1
    
    return annualized_return

In [7]:
def run_backtest(df, initial_cash=100000, commission=0.001):
    """
    运行回测函数
    
    参数:
    df: 包含以下列的DataFrame: open, high, low, close, volume, long, stop_loss_long, sell
    initial_cash: 初始资金
    commission: 交易佣金
    """
    
    cerebro = bt.Cerebro()
    
    # 检查数据格式
#     print("数据前5行:")
#     print(df.head())
#     print("\n数据信息:")
#     print(df.info())
    
    # 确保索引是datetime类型
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    
    # 创建数据feed
    data = CustomData(
        dataname=df,
        open='open',
        high='high',
        low='low',
        close='close',
        volume='volume',
        long='long',
        stop_loss_long='stop_loss_long',
        sell='sell',
        timeframe=bt.TimeFrame.Minutes,  # 设置为分钟线
        compression=5,  # 周期为5分钟
        openinterest=None
    )
    
    cerebro.adddata(data)
  
    # 添加策略
    cerebro.addstrategy(LongStrategy)
    
    # 设置初始资金
    cerebro.broker.setcash(initial_cash)
    
    # 设置佣金
    cerebro.broker.setcommission(commission=commission)
    
    # 添加分析器
    cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe', riskfreerate=0.0)
    cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')
    cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
    cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trades')
#     cerebro.addanalyzer(bt.analyzers.AnnualReturn, _name='annualreturn')
    cerebro.addanalyzer(AnnualReturnAnalyzer, _name='_AnnualReturn')

    print('=' * 50)
    print('开始回测')
    print('=' * 50)
    print(f'初始资金: {initial_cash:,.2f}')
    
    # 运行回测
    results = cerebro.run()
    strat = results[0]
    
    # 打印最终结果
    print('=' * 50)
    print('回测结果')
    print('=' * 50)
    print(f'最终资金: {cerebro.broker.getvalue():,.2f}')
    print(f'总收益率: {(cerebro.broker.getvalue()/initial_cash-1)*100:.2f}%')
    
    # 分析结果
#     sharpe = strat.analyzers.sharpe.get_analysis()
#     drawdown = strat.analyzers.drawdown.get_analysis()
#     returns = strat.analyzers.returns.get_analysis()
#     trades = strat.analyzers.trades.get_analysis()

#     print(f"夏普比率: {sharpe.get('sharperatio', 0):.3f}")
#     print(f"最大回撤: {drawdown['max']['drawdown']:.2f}%")
#     print(f"年化收益率: {returns.get('rnorm100', 0):.2f}%")

    # 安全获取分析结果
    sharpe_analysis = safe_get_analysis(strat.analyzers.sharpe)
    drawdown_analysis = safe_get_analysis(strat.analyzers.drawdown)
    trades_analysis = safe_get_analysis(strat.analyzers.trades)
    returns_analysis = safe_get_analysis(strat.analyzers.returns)
#     returns_analysis = safe_get_analysis(strat.analyzers.annualreturn)
    returns_analysis = strat.analyzers._AnnualReturn.get_analysis()

    trades = strat.analyzers.trades.get_analysis()

    # 安全打印夏普比率
    sharpe_ratio = sharpe_analysis.get('sharperatio', 0)
    print(f"夏普比率: {format_value(sharpe_ratio, '.3f')}")    
    # 安全打印最大回撤
    max_drawdown = drawdown_analysis.get('max', {}).get('drawdown', 0)
    print(f"最大回撤: {format_value(max_drawdown, '.2f')}%")  
    # 安全打印年化收益率
    annual_return = returns_analysis.get('annual_return', 0)
    start = returns_analysis.get('start_date', 0)
    end = returns_analysis.get('end_date', 0)
    years = returns_analysis.get('total_years', 0)
    print(f"从{start}开始，{end}结束, 共{format_value(years,'.2f')}年,年化收益率: {format_value(annual_return*100, '.2f')}%")
    
    # 交易统计
    if 'total' in trades:
        print(f"总交易次数: {trades['total']['total']}")
        if trades['total']['total'] > 0:
            print(f"盈利交易比例: {trades['won']['total']/trades['total']['total']*100:.1f}%")
            print(f"平均每笔利润: {trades['pnl']['net']['average']:.2f}")
    
    # 启用交互模式
    plt.ion()
    # 绘制图表
    print('\n正在生成图表...')
    cerebro.plot(style='candlestick', volume=True)

# # 使用示例
# if __name__ == '__main__':
#     # 创建示例数据（请用您自己的数据替换这部分）
#     dates = pd.date_range(start='2023-01-01', end='2023-12-31', freq='D')
#     n_points = len(dates)
    
#     # 生成价格数据
#     np.random.seed(42)
#     prices = [100]
#     for i in range(n_points-1):
#         change = np.random.normal(0, 2)
#         new_price = prices[-1] + change
#         prices.append(max(new_price, 1))  # 确保价格不为负
    
#     # 创建示例DataFrame
#     sample_df = pd.DataFrame({
#         'open': [p * 0.99 for p in prices],
#         'high': [p * 1.02 for p in prices],
#         'low': [p * 0.98 for p in prices],
#         'close': prices,
#         'volume': np.random.randint(1000, 10000, n_points),
#         'long': np.random.choice([0, 1], n_points, p=[0.85, 0.15]),
#         'stop_loss_long': [p * 0.95 for p in prices],  # 止损价为收盘价的95%
#         'sell': np.random.choice([0, 1], n_points, p=[0.9, 0.1])
#     }, index=dates)
    
#     # 运行回测
#     run_backtest(sample_df, initial_cash=100000, commission=0.001)

In [ ]:
def prepare_back_test_data(path,para):
    raw = pd.read_csv(path)
    klines=raw.loc[:,['time','open','high','low','close','volume']]
    base = calc_base_data(klines)
    base['long'] = gen_long_signal(base,para)
    base['stop_loss_long'] = gen_stop_loss_long(base,para)
    base['sell'] = 0
    base.set_index('time',inplace=True)
    base.index = pd.to_datetime(base.index)
    return base